CLASIFICACION
JOAN CAMILO GALVIS BARRANCO

1. Preparación de datos
2. División de los datos
3. Aprendizaje del modelo
4. Evaluación del modelo
5. Guardar el modelo

In [ ]:
#imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
#carga de datos
path = "https://github.com/camilojoanbarrancogalvis-debug/colab/raw/refs/heads/main/adult_clean.csv"
df = pd.read_csv(path)
df.head()

####1. Preparación de datos

In [ ]:
df.info()

In [ ]:
#correcion de tipos
for col in df.columns.tolist():
  if df[col].dtype != np.int64:
    df[col] = df[col].astype('category')

df.info()

In [ ]:
df.describe()

In [ ]:
df.plot(kind = "box")

In [ ]:
!pip install ydata-profiling

In [ ]:
#Analisis descriptivo
from ydata_profiling import ProfileReport
ProfileReport(df)

In [ ]:
#configuracion
data = df.copy()
tarjet = 'income'
numericas = ['age', 'capital-gain', 'capital-loss', 'hours-per-week']
categoricas_binarias = ['sex']
categoricas_narias = ['workclass', 'marital-status', 'education', 'occupation']
data['income'].value_counts().plot(kind="bar")

In [ ]:
#balanceo
from imblearn.over_sampling import SMOTE, SMOTENC
X = data.drop(columns=[tarjet])
Y = data[tarjet]
smote = SMOTENC(k_neighbors=2, categorical_features=[1,2,3,4,5], sampling_strategy=0.4)
X_smote, Y_smote = smote.fit_resample(X,Y)
data = X_smote
data[tarjet] = Y_smote
data[tarjet].value_counts().plot(kind="bar")

In [ ]:
#normalizacion
from sklearn.preprocessing import MinMaxScaler
min_max_scaler = MinMaxScaler()
data[numericas] = min_max_scaler.fit_transform(data[numericas])
data.head()

In [ ]:
#normalizacion objetivo
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder() #label_encoder: le
objetivo = le.fit_transform(data[tarjet])
data[tarjet] = objetivo
data.head()

In [ ]:
#codificacion
data = pd.get_dummies(data, columns = categoricas_binarias, drop_first=True, dtype = int)
data = pd.get_dummies(data, columns = categoricas_narias, drop_first=False, dtype = int)
data.head()

####2. División de datos

In [ ]:
#Division de datos 70 - 30
from sklearn.model_selection import train_test_split
X = data.drop(columns = [tarjet])
Y = data[tarjet]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.3,
    random_state=46,
    stratify=Y
)

####3. Modelamiento

#####Tree

In [ ]:
from scipy.stats import randint
from sklearn import tree
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, recall_score, precision_score, roc_auc_score
#modelo base
model_tree = tree.DecisionTreeClassifier(criterion = 'gini', random_state = 46)
#hiperparametros
parametros = {
    'max_depth': randint(2,20),
    'min_samples_leaf': randint(2,60),
    'min_samples_split': randint(2,60)
}
#busqueda de hiperparametros
random_search = RandomizedSearchCV(
    estimator = model_tree,
    param_distributions = parametros,
    n_iter = 30,
    scoring = 'f1_macro',
    random_state = 46,
    cv = 10,
    n_jobs = -1
)
random_search.fit(X_train, y_train)
print(random_search.best_score_)
print(random_search.best_params_)

In [ ]:
#evaluacion
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
#Medidas
print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names=le.classes_)}")

In [ ]:
#matriz de confusion
sns.heatmap(data = confusion_matrix(y_test, y_pred), annot = True, fmt = 'd', xticklabels=le.classes_,
    yticklabels=le.classes_, cmap ='Blues')

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(60,50))
plot_tree(best_model, feature_names=X.columns.values, class_names=le.classes_, rounded=True, filled=False, fontsize = 8)
plt.show()

In [ ]:
importancias = pd.DataFrame({
    "variable": X.columns,
    "importancia": best_model.feature_importances_
}).sort_values("importancia", ascending=False)
importancias.head()

In [ ]:
comparacion = []
def resultados(lista, model, y_pred):
  lista.append({'modelo': model,
             'f1': f1_score(y_test, y_pred, average='macro'),
             'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='macro'),
            'recall': recall_score(y_test, y_pred, average='macro'),
            'roc': roc_auc_score(y_test, y_pred, average='macro')})
resultados(comparacion, 'tree', y_pred)
print(comparacion)

#####Knn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
#modelo base
model_knn = KNeighborsClassifier(metric="euclidean")
#parametros
parametros = {
    'n_neighbors': [1,3,5,7,9]
}
grid = GridSearchCV(
    estimator = model_knn,
    param_grid = parametros,
    cv = 10,
    n_jobs = -1,
    scoring = 'f1_macro'
)
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

In [ ]:
#evaluacion
best_model_knn = grid.best_estimator_
y_pred = best_model_knn.predict(X_test)
#medidas
print(f"Medidas de evaluacion: \n{classification_report(y_test, y_pred)}")
#matriz de confusion
plt.figure()
sns.heatmap(data=confusion_matrix(y_test, y_pred), cmap ="Blues", annot = True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.show()
#guardar medidas
resultados(comparacion, 'knn', y_pred)

#####Red Neuronal

In [ ]:
from sklearn.neural_network import MLPClassifier
#modelo base
model_nn = MLPClassifier(hidden_layer_sizes=(8,6),learning_rate='adaptive', learning_rate_init=0.1,
                         random_state = 46, alpha=0.001, momentum=0.2, max_iter = 500)
model_nn.fit(X_train, y_train)

In [ ]:
#evaluacion
y_pred = model_nn.predict(X_test)
print(f"Medidas de evaluación: \n {classification_report(y_test, y_pred,target_names=le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), cmap='Blues', annot = True, fmt = 'd', xticklabels=le.classes_, yticklabels=le.classes_)
plt.show()
#guardar datos
resultados(comparacion, 'Red', y_pred)

Máquina de soporte vectorial

In [ ]:
from sklearn.svm import SVC
model_svm = SVC(kernel='rbf', C=5)
model_svm.fit(X_train, y_train)
y_pred = model_svm.predict(X_test)
print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names=le.classes_)}")
plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), annot = True, cmap="Blues", fmt = 'd', xticklabels=le.classes_, yticklabels=le.classes_)
plt.show()
resultados(comparacion, 'svm', y_pred)

Regresion logistica

In [ ]:
from sklearn.linear_model import LogisticRegression

model_logistic = LogisticRegression()
model_logistic.fit(X_train, y_train)

y_pred = model_logistic.predict(X_test)
print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred,  target_names = le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), xticklabels=le.classes_, yticklabels=le.classes_, annot = True, cmap="Blues", fmt="d")
plt.show()

Bagging Knn

In [ ]:
#Bagging: Knn
from sklearn.ensemble import BaggingClassifier

#modelo base
modelo_base = KNeighborsClassifier(metric = 'euclidean', n_neighbors=7)

#modelo bagging
model_bag = BaggingClassifier(modelo_base, n_estimators=100, max_samples=0.6)
model_bag.fit(X_train, y_train)

#Evaluación
y_pred = model_bag.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

plt.figure(figsize=(10,6))
sns.heatmap(confusion_matrix(y_test, y_pred), cmap = "Blues", annot = True, xticklabels=le.classes_, yticklabels=le.classes_, fmt = 'd')
plt.show()

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(n_estimators=100, max_samples=0.7, criterion = 'gini', max_depth=None, min_samples_leaf=2)
model_rf.fit(X_train, y_train)
y_pred = model_rf.predict(X_test)

print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names=le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), annot = True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.show()

resultados(comparacion, 'rf', y_pred  )

In [ ]:
#importancias
importancias = pd.DataFrame({
    'variables': X.columns.values,
    'importancia': model_rf.feature_importances_
}).sort_values(by = 'importancia', ascending = False)
importancias.head()

Boosting

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
model_base = tree.DecisionTreeClassifier(min_samples_leaf = 3, criterion = 'gini', max_depth=None)
model_ada = AdaBoostClassifier(model_base, n_estimators = 100)
model_ada.fit(X_train, y_train)
y_pred = model_ada.predict(X_test)

print(f"Medidas de evaluacion: \n{classification_report(y_test, y_pred, target_names = le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), xticklabels=le.classes_, yticklabels=le.classes_, annot = True, fmt = 'd', cmap='Blues')
plt.show()

resultados(comparacion, 'ada', y_pred)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
model_gbc = GradientBoostingClassifier(n_estimators = 100, learning_rate = 0.1, min_samples_leaf = 2, max_depth = None, subsample = 0.8)
model_gbc.fit(X_train, y_train)
y_pred = model_gbc.predict(X_test)

print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names=le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues', annot = True, fmt = 'd')
plt.show()
resultados(comparacion, 'gbc', y_pred)

In [ ]:
import xgboost as xgb

model_xgb = xgb.XGBClassifier(
    max_depth = 15,
    learning_rate = 0.1,
    n_estimators = 100
)

model_xgb.fit(X_train, y_train)
y_pred = model_xgb.predict(X_test)

print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names = le.classes_)}")
resultados(comparacion, 'xgb', y_pred)
plt.figure(figsize=(10,6))
sns.heatmap(data=confusion_matrix(y_test, y_pred), xticklabels=le.classes_, yticklabels=le.classes_, cmap = "Blues", annot = True, fmt='d')
plt.show()

Votación Hard

In [ ]:
from sklearn.ensemble import VotingClassifier
clasificadores = [('xgb', model_xgb), ('bag', model_bag), ('rf', model_rf)]
model_vot_hard = VotingClassifier(estimators = clasificadores, voting = 'hard')
model_vot_hard.fit(X_train, y_train)
y_pred = model_vot_hard.predict(X_test)

print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names=le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), annot = True, fmt ='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.show()
resultados(comparacion, 'vot_hard', y_pred)

Votacion soft

In [ ]:
clasificadores = [('xgb', model_xgb),('bagg', model_bag),('ada', model_ada)]
model_vot_soft = VotingClassifier(estimators = clasificadores, voting = 'soft', weights=[0.6,0.3,0.1])
model_vot_soft.fit(X_train, y_train)
y_pred = model_vot_soft.predict(X_test)

print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names=le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), annot = True, fmt ='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.show()
resultados(comparacion, 'vot_sof', y_pred)

In [ ]:
from sklearn.ensemble import StackingClassifier
clasificadores = [('xgb', model_xgb),('bagg', model_ada),('soft', model_vot_soft)]
ensamblador = RandomForestClassifier()

model_stacking = StackingClassifier(estimators = clasificadores, final_estimator=ensamblador)
model_stacking.fit(X_train, y_train)
y_pred = model_stacking.predict(X_test)

print(f"Medidas de evaluación: \n{classification_report(y_test, y_pred, target_names=le.classes_)}")

plt.figure(figsize=(10,6))
sns.heatmap(data = confusion_matrix(y_test, y_pred), annot = True, fmt ='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.show()
resultados(comparacion, 'stacking', y_pred)

In [ ]:
resultado = pd.DataFrame(comparacion).sort_values(by='f1', ascending = False)
resultado.head()

In [ ]:
import pickle
filename = 'modelo-class.pkl'
variables = X.columns
pickle.dump([model_xgb, min_max_scaler, variables], open(filename, 'wb'))